# 🔥 LLM Fine-tuning with QLoRA (LoRA + 4-bit Quantization)

**Fine-tune microsoft/phi-2 on the Alpaca instruction-following dataset using:**
-  4-bit NF4 Quantization (BitsAndBytes)
-  LoRA Adapters (r=8, alpha=32)
-  SFTTrainer (TRL)
-  paged_adamw_8bit optimizer
-  Weights & Biases logging

> **Runtime:** GPU → Runtime > Change runtime type > T4 GPU (Free)


## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers peft bitsandbytes datasets accelerate trl wandb sentencepiece einops

## 2. Check GPU

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Model
    model_name: str = 'microsoft/phi-2'
    output_dir: str = './qlora-phi2-output'

    # 4-bit QLoRA quantization
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = 'nf4'       # NormalFloat4 — best for QLoRA
    use_double_quant: bool = True           # Nested quantization

    # LoRA adapter
    lora_r: int = 8                         # Low-rank dimension
    lora_alpha: int = 32                    # Scale = alpha/r = 4
    lora_dropout: float = 0.05
    target_modules: list = None

    # Training
    num_epochs: int = 3
    batch_size: int = 4
    grad_accum: int = 4                     # Effective batch = 16
    lr: float = 2e-4
    max_seq_len: int = 512
    max_samples: int = 5000

    def __post_init__(self):
        self.target_modules = ['q_proj', 'k_proj', 'v_proj', 'dense']

cfg = Config()
print('Config ready:', cfg)

## 4. Load Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# 4-bit NF4 quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

print(f'Loading {cfg.model_name} with 4-bit quantization...')
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Attach LoRA adapters
lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.target_modules,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% of params — that's the LoRA magic!

## 5. Prepare Dataset

In [ ]:
from datasets import load_dataset

def format_alpaca(sample):
    instruction = sample.get('instruction', '')
    input_text = sample.get('input', '')
    output = sample.get('output', '')
    if input_text:
        return {'text': f'### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}'}
    return {'text': f'### Instruction:\n{instruction}\n\n### Response:\n{output}'}

dataset = load_dataset('tatsu-lab/alpaca', split='train')
dataset = dataset.select(range(cfg.max_samples))
dataset = dataset.map(format_alpaca, remove_columns=dataset.column_names)

print(f'Dataset: {len(dataset)} samples')
print('\nSample:')
print(dataset[0]['text'][:300])

## 6. Train with SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to='none',
    group_by_length=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=cfg.max_seq_len,
    dataset_text_field='text',
    packing=True,
)

print('Starting QLoRA fine-tuning...')
trainer.train()

## 7. Plot Training Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
steps = [l['step'] for l in logs if 'loss' in l]
losses = [l['loss'] for l in logs if 'loss' in l]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, color='orange', linewidth=2, label='Train Loss')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('QLoRA Fine-tuning: Phi-2 on Alpaca Dataset (5K samples)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print(f'Final loss: {losses[-1]:.4f}')

## 8. Save & Test Inference

In [ ]:
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f'Model saved to {cfg.output_dir}')

# Quick inference 
model.eval()
test_prompt = '### Instruction:\nExplain what a transformer model is in simple terms.\n\n### Response:\n'
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

response = tokenizer.decode(out[0], skip_special_tokens=True)
print('\n=== Inference Test ===')
print(response.split('### Response:')[-1].strip())